# Dataset Verification
Verify YOLO conversion is correct: visualize bounding boxes over images.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

YOLO_ROOT = Path('../data/raw/plates')
SPLIT = 'train'  # change to 'val' or 'test'

In [ ]:
def yolo_to_abs(cx, cy, bw, bh, img_w, img_h):
    x1 = int((cx - bw / 2) * img_w)
    y1 = int((cy - bh / 2) * img_h)
    x2 = int((cx + bw / 2) * img_w)
    y2 = int((cy + bh / 2) * img_h)
    return x1, y1, x2, y2

img_dir = YOLO_ROOT / 'images' / SPLIT
lbl_dir = YOLO_ROOT / 'labels' / SPLIT

images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
sample = random.sample(images, min(8, len(images)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            cls_id, cx, cy, bw, bh = parts
            x1, y1, x2, y2 = yolo_to_abs(cx, cy, bw, bh, w, h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(img)
    ax.set_title(img_path.name[:30], fontsize=7)
    ax.axis('off')
plt.suptitle(f'Split: {SPLIT} — bounding boxes from YOLO labels', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Dataset stats
for split in ['train', 'val', 'test']:
    imgs = list((YOLO_ROOT / 'images' / split).glob('*.jpg'))
    lbls = list((YOLO_ROOT / 'labels' / split).glob('*.txt'))
    print(f"{split:6s}  images: {len(imgs):4d}  labels: {len(lbls):4d}")